In [1]:
import pandas as pd


In [2]:
df = pd.read_csv("IMDB Dataset.csv")

In [3]:
df.shape

(50000, 2)

In [4]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [5]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [6]:
df.drop_duplicates(inplace=True)

In [7]:
df.shape

(49582, 2)

## Pre-processing

### 1. converting to lowercase

In [8]:
df["review"] = df["review"].str.lower()

### 2. Removing the URLs

In [9]:
import re

In [10]:
def remove_urls(text):
    text = re.sub(r"http\S+", "", text)
    return text

df["review"] = df["review"].apply(remove_urls)

### 3. Removing punchuation

In [11]:
def remove_punctuations(text):
    text = re.sub(r"[^A-Za-z0-9\s]", "", text)
    return text

df["review"] = df["review"].apply(remove_punctuations)

### 4. Removing HTML

In [12]:
def remove_html(text):
    text = re.sub(r"<,*?>", "", text)
    return text

df["review"] = df["review"].apply(remove_html)

### 5. Removing the stopwords

In [13]:
import nltk

In [14]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [15]:
sample_text = "I like coding in python!"
tokens = word_tokenize(sample_text)

In [16]:
tokens

['I', 'like', 'coding', 'in', 'python', '!']

In [17]:
def remove_stopwords(text):
    tokens = word_tokenize(text)
    stop_words = stopwords.words("english")

    for word in tokens:
        if word in stop_words:
            text = text.replace(word, "")

    return text

df["review"] = df["review"].apply(remove_stopwords)

In [18]:
df.head()

,review,sentiment
0,e revewers nted wtchg 1 oz epode ll ho...,positive
1,wderful ltle producti br br filming techniqu...,positive
2,thought ths wderful wy spend tme o hot s...,positive
3,bsclly res fmly lttle boy jke thks res zom...,negative
4,petter mtte love time mey vully stunng fi...,positive


### 6. Stemming

In [19]:
from nltk.stem import PorterStemmer

In [21]:
def stemming(text):
    ps = PorterStemmer()
    stemmed_words = []

    tokens = word_tokenize(text)
    for token in tokens:
        stemmed_token = ps.stem(token)
        stemmed_words.append(stemmed_token)

    return " ".join(stemmed_words)

df["review"] = df["review"].apply(stemming)

In [22]:
df.head()

,review,sentiment
0,e revew nted wtchg 1 oz epod ll hook y rght ex...,positive
1,wder ltle producti br br film techniqu unssum ...,positive
2,thought th wder wy spend tme o hot summer week...,positive
3,bsclli re fmli lttle boy jke thk re zomb close...,negative
4,petter mtte love time mey vulli stunng film wt...,positive


### 7. Encoding

In [24]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df["sentiment"] = le.fit_transform(df["sentiment"])

In [25]:
y = df["sentiment"]

In [26]:
y

0        1
1        1
2        1
3        0
4        1
        ..
49995    1
49996    0
49997    0
49998    0
49999    0
Name: sentiment, Length: 49582, dtype: int64

### 8. Vectorization

In [27]:
from sklearn.feature_extraction.text import TfidfVectorizer

tf = TfidfVectorizer(max_features = 5000)

X = tf.fit_transform(df["review"])

## Dataset & Data Loaders

In [32]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size = 0.2, random_state = 42
)

In [33]:
X_train.shape

(39665, 5000)

In [34]:
X_test.shape

(9917, 5000)

In [43]:
import torch
from torch.utils.data import TensorDataset, DataLoader

In [46]:
train_set = TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train.values).float()
)

test_set = TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test.values).float()
)

In [47]:
train_loader = DataLoader(train_set, shuffle = True, batch_size = 64)
test_loader = DataLoader(test_set, shuffle = True, batch_size = 64)

## Build our RNN

In [57]:
import torch.nn as nn
import torch.optim as optim

In [58]:
class RNN(nn.Module):
    def __init__(self, input_size, hidden_size = 128, num_layers = 1):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        #RNN layer
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first = True)

        # fully connected Layer
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.rnn(x)

        out = self.fc(out[:, -1, :])
        return out

In [62]:
input_size = X_train.shape[1]

model = RNN(input_size)

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters())

## Training the RNN

In [64]:
epochs = 10

for epoch in range(epochs):
    model.train()

    for Xb, yb in train_loader:
        optimizer.zero_grad()

        Xb = Xb.unsqueeze(1)

        outputs = model(Xb)

        outputs = torch.sigmoid(outputs.squeeze())

        loss = criterion(outputs, yb)
        loss.backward()
        optimizer.step()

    print(f"epoch = {epoch + 1}/{epochs} and loss = {loss.item()}")

epoch = 1/10 and loss = 0.2170264720916748
epoch = 2/10 and loss = 0.1885247677564621
epoch = 3/10 and loss = 0.18325836956501007
epoch = 4/10 and loss = 0.2385970503091812
epoch = 5/10 and loss = 0.18810398876667023
epoch = 6/10 and loss = 0.23550045490264893
epoch = 7/10 and loss = 0.1755317598581314
epoch = 8/10 and loss = 0.32156646251678467
epoch = 9/10 and loss = 0.20576612651348114
epoch = 10/10 and loss = 0.1654597371816635


## Evaluate

In [67]:
model.eval()

with torch.no_grad():
    correct_vals = 0
    tot_vals = 0

    for Xb, yb in test_loader:
        Xb = Xb.unsqueeze(1)

        outputs = model(Xb)
        predicted = (torch.sigmoid(outputs.squeeze()) > 0.5).float()

        tot_vals += yb.size(0)
        correct_vals += (predicted == yb).sum().item()

    print(f"Accuracy = {correct_vals/tot_vals * 100}")

Accuracy = 85.4593122920238
